# Lesson 11: Assemble a pre-LayerNorm Transformer block

A Transformer block alternates **communication** across token positions and **computation** within each token. The attention branch lets tokens gather information from the past. The feed-forward branch then transforms each token's channels independently.

Karpathy's block uses two pre-LayerNorm residual sublayers:

1. Add multi-head attention applied to LayerNorm(input).
2. Add a feed-forward network applied to LayerNorm(after attention).

The C and Java examples use the same tiny input and a fixed attention-output tensor calculated by Lesson 10 on the normalized input. This keeps the lesson focused on the new residual and feed-forward structure instead of duplicating the attention implementation.

For the matching implementations, start at [the C entry point](../c/src/transformer_block_main.c) and [the Java entry point](../java/src/main/java/learning/karpathy/gpt1/TransformerBlockLesson.java). The Java file contains a small FeedForward helper class for its two learned linear transforms.

## Set up the shared example

The model width is 4, so the feed-forward network expands each token to 16 channels, applies ReLU, then projects back to 4. LayerNorm uses the last dimension and begins with gamma=1 and beta=0. Feed-forward weights are small deterministic examples so all three language versions agree.

The attention result is the recorded output of the Lesson 10 multi-head example on LayerNorm(input), using the same shared attention weights. It is an explicit fixture, so this notebook and the array implementations stay focused on the Transformer block composition.

In [ ]:
import torch
from torch import nn

torch.set_printoptions(precision=4, sci_mode=False)

BATCH_SIZE = 2
TIME_STEPS = 4
CHANNELS = 4
HIDDEN_SIZE = 4 * CHANNELS
DROPOUT = 0.2

x = torch.tensor(
    [
        [
            [1.0, 0.0, 2.0, -1.0],
            [0.0, 1.0, 1.0, 2.0],
            [2.0, -1.0, 0.0, 1.0],
            [1.0, 2.0, -1.0, 0.0],
        ],
        [
            [0.0, 2.0, 1.0, -1.0],
            [1.0, -1.0, 2.0, 0.0],
            [-1.0, 0.0, 1.0, 2.0],
            [2.0, 1.0, 0.0, -1.0],
        ],
    ],
    dtype=torch.float64,
)

# Output of Lesson 10 multi-head attention on LayerNorm(x).
attention_output = torch.tensor(
    [
        [
            [0.9944236133126180, -0.0881970483359226, 0.05, -1.6006501782893132],
            [0.3933738827285683, 0.1954140878650754, 0.2038379239277406, -0.6362878460652932],
            [-0.5175716661695421, -1.3505289024072524, 1.4923256127023328, -0.1986207426936189],
            [0.6142816558095826, -0.5870337399944243, -0.4857832613323823, 1.4064524206079174],
        ],
        [
            [0.3236059033281545, -0.5354088549922318, -0.6208177099844634, 1.0826206616485405],
            [0.6468403211190350, 0.2777975158844621, 0.5677887061817602, -1.9983469601638093],
            [0.6771942604404187, -0.9952272543893472, 0.1465132161629877, -0.0665467975441047],
            [-0.2972420175988929, 0.2774110104487157, 0.7685343078948748, -0.3120942225247459],
        ],
    ],
    dtype=torch.float64,
)

layer_norm_attention = nn.LayerNorm(CHANNELS, dtype=torch.float64)
layer_norm_feed_forward = nn.LayerNorm(CHANNELS, dtype=torch.float64)

first_linear = nn.Linear(CHANNELS, HIDDEN_SIZE, dtype=torch.float64)
second_linear = nn.Linear(HIDDEN_SIZE, CHANNELS, dtype=torch.float64)
feed_forward = nn.Sequential(
    first_linear,
    nn.ReLU(),
    second_linear,
    nn.Dropout(DROPOUT),
)

# Use simple deterministic weights instead of random initialization for parity.
with torch.no_grad():
    for hidden in range(HIDDEN_SIZE):
        first_linear.bias[hidden] = (hidden % 3 - 1) * 0.05
        for channel in range(CHANNELS):
            pattern = ((hidden + 1) * (channel + 2)) % 7 - 3
            first_linear.weight[hidden, channel] = pattern * 0.1

    for channel in range(CHANNELS):
        second_linear.bias[channel] = [0.02, -0.01, 0.03, -0.02][channel]
        for hidden in range(HIDDEN_SIZE):
            pattern = ((channel + 2) * (hidden + 1)) % 5 - 2
            second_linear.weight[channel, hidden] = pattern * 0.05

feed_forward.eval()  # Dropout is inactive for deterministic cross-language comparison.

print("Input shape:", tuple(x.shape))
print("Feed-forward widths:", CHANNELS, "->", HIDDEN_SIZE, "->", CHANNELS)
print("Attention output shape:", tuple(attention_output.shape))

## Step 1: Normalize each token before attention

LayerNorm computes statistics across each token's channels, independently for every batch item and time position. Karpathy's block normalizes the input before the attention sublayer; the residual branch still carries the original, unnormalized input.

In [ ]:
with torch.no_grad():
    normalized_input = layer_norm_attention(x)

print("LayerNorm(input), shape:", tuple(normalized_input.shape))
print("First token before:", x[0, 0])
print("First token after: ", normalized_input[0, 0])
print("Per-token means (approximately zero):", normalized_input.mean(dim=-1))
print("Per-token variances (approximately one):", normalized_input.var(dim=-1, unbiased=False))

## Step 2: Add the attention residual

The first residual connection adds the attention branch's output to the original representation:

x_after_attention = x + attention_output

The original signal therefore has a direct path through the block. Here the attention tensor is the fixed Lesson 10 output produced from normalized_input.

In [ ]:
with torch.no_grad():
    after_attention = x + attention_output

print("Attention branch output, batch 0:\n", attention_output[0])
print("After residual addition, batch 0:\n", after_attention[0])
assert after_attention.shape == x.shape

## Step 3: Normalize, expand, activate, and contract

The feed-forward network is applied independently to each token vector. Its first linear layer expands the width from 4 to 16, ReLU adds a nonlinearity, and the second linear layer returns to width 4. Dropout follows the second layer; in this evaluation example it acts as identity.

In [ ]:
with torch.no_grad():
    normalized_after_attention = layer_norm_feed_forward(after_attention)
    hidden = feed_forward[1](feed_forward[0](normalized_after_attention))
    feed_forward_output = feed_forward[3](feed_forward[2](hidden))

print("LayerNorm output shape:", tuple(normalized_after_attention.shape))
print("Expanded hidden shape:", tuple(hidden.shape))
print("First token after ReLU:", hidden[0, 0])
print("Feed-forward output shape:", tuple(feed_forward_output.shape))
print("Feed-forward output, batch 0:\n", feed_forward_output[0])
assert hidden.shape == (BATCH_SIZE, TIME_STEPS, HIDDEN_SIZE)
assert feed_forward_output.shape == x.shape

## Step 4: Add the second residual

The block adds the feed-forward result to the representation that entered that sublayer. The block preserves batch, time, and channel dimensions, so another block can consume the same shape.

In [ ]:
with torch.no_grad():
    block_output = after_attention + feed_forward_output

print("Transformer block output shape:", tuple(block_output.shape))
print("Block output, batch 0:\n", block_output[0])

# The feed-forward network is position-wise: changing one token cannot change
# the feed-forward result at another time position.
with torch.no_grad():
    changed_normalized = normalized_after_attention.clone()
    changed_normalized[0, 3, 0] += 10.0
    changed_hidden = feed_forward[1](feed_forward[0](changed_normalized))
    changed_feed_forward = feed_forward[3](feed_forward[2](changed_hidden))
    other_token_difference = (
        feed_forward_output[0, :3] - changed_feed_forward[0, :3]
    ).abs().max()
print(f"Change at other token positions: {other_token_difference.item():.12f}")

torch.testing.assert_close(feed_forward_output[0, :3], changed_feed_forward[0, :3], rtol=0, atol=1e-12)
assert block_output.shape == x.shape
print("Check passed: the feed-forward network mixes channels, not positions.")

## What to notice

Attention communicates across time; the feed-forward network transforms the channels at one time position. Residual additions preserve a path for the incoming representation, and LayerNorm makes each sublayer's input scale predictable. The block's output shape stays the same as its input shape.

The C and Java companions implement LayerNorm, the residual additions, and both feed-forward linear layers with arrays and loops. Their attention branch is the same fixed intermediate from Lesson 10, and dropout is inactive in this evaluation pass. Next we add token and position embeddings, the inputs that repeated Transformer blocks will use in a full decoder-only GPT.

References: [PyTorch LayerNorm](https://docs.pytorch.org/docs/2.14/generated/torch.nn.LayerNorm.html), [PyTorch Sequential](https://docs.pytorch.org/docs/2.14/generated/torch.nn.Sequential.html), [PyTorch Dropout](https://docs.pytorch.org/docs/2.14/generated/torch.nn.Dropout.html), and Karpathy's [original notebook](../gpt_dev.ipynb).